## Para utilizar en la clase 03, del 30-mayo-2025
##  **NO UTILIZAR ANTES**

# AutoGluon
Matar una mosca con una  bazooka, entrena automaticamente modelos de:


*   Estadistica Clasica
*   Machine Learning
*   Deep Learning


 ['SeasonalNaive', 'RecursiveTabular', 'DirectTabular', 'DynamicOptimizedTheta', 'Chronos2', 'Chronos2SmallFineTuned', 'AutoETS', 'ChronosWithRegressor[bolt_small]', 'TemporalFusionTransformer', 'DeepAR']



## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental, POR UNICA VEZ, seguir los siguientes pasos

* Registrar usuario en Kaggle con la cuenta de email de la Universidad Austral
* Hacer el "Join Competition"  a la competencia de  Labo 3
* Generar el archivo kaggle.json  a partir de   https://www.kaggle.com/settings/account  y presione  "Create Legacy API Key"
* Crear carpeta  labo3  en  el Google Drive
* Dentro de la carpeta labo3 crear carpeta   kaggle
* Subir a la carpeta kaggle el archivo  kaggle.json


In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"


# 1  Modelo AutoGluon

## 1.1 Init Experimento

In [15]:
# instalacion de paquetes que NO vienen por default en Colab
# autogluon es MUY pesado. varios minutos se perderan aqui
!pip install uv
!uv pip install -q kaggle
!uv pip install autogluon[all]


Using Python 3.12.13 environment at: /usr
Resolved 247 packages in 405ms
Prepared 79 packages in 2m 18s
Uninstalled 12 packages in 1.98s
Installed 84 packages in 991ms
 + adagio==0.2.6
 + aiohttp-cors==0.8.1
 + autogluon==1.5.0
 + autogluon-common==1.5.0
 + autogluon-core==1.5.0
 + autogluon-features==1.5.0
 + autogluon-multimodal==1.5.0
 + autogluon-tabular==1.5.0
 + autogluon-timeseries==1.5.0
 + boto3==1.43.23
 + botocore==1.43.23
 + catboost==1.2.10
 + chronos-forecasting==2.2.2
 + colorama==0.4.6
 + colorful==0.5.8
 + colorlog==6.10.1
 + coreforecast==0.0.16
 + distlib==0.4.1
 + einx==0.4.3
 + evaluate==0.4.6
 + fugue==0.9.7
 + gluonts==0.16.2
 - huggingface-hub==1.17.0
 + huggingface-hub==0.36.2
 + jmespath==1.1.0
 - jsonschema==4.26.0
 + jsonschema==4.23.0
 + lightning==2.5.6
 + lightning-utilities==0.15.3
 + loguru==0.7.3
 + mlforecast==0.14.0
 + model-index==0.1.11
 + nlpaug==1.1.11
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==1

In [16]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [17]:
import os as os
import numpy as np
import polars as pl

from datetime import datetime
from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame

Por favor, cargar aqui SU semilla primigenia
<br> **Muy importante**, cambiar el numero de experimento en cada corrida. Usted ha sido notificado !
<br> Si cada corrida no está en una nueva carpeta virgen, entonces se reutilizarn modelos viejos corridos con otros parametros
https://www.youtube.com/shorts/0_0Kzqpdn1o

In [18]:
# defino los parametros
PARAM = {'experimento':'AutoGluon-02',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':  19970220
}

In [8]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AutoGluon-01


## 1.2 Init AutoGluon

In [9]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [10]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [13]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [20]:
# NUEVO: cargar productos y hacer join
tb_productos = pl.read_csv('/content/.drive/My Drive/labo3/datasets/tb_productos.txt', separator="\t")
tb_ventas = tb_ventas.join(tb_productos, on="product_id", how="left")

In [21]:
# paso de periodo a  timestamp
tb_ventas = tb_ventas.with_columns(
    (pl.col('periodo').cast(pl.String).str.to_datetime('%Y%m')).alias('timestamp')
)

Opcion de *empiojar el dataset*
<br>agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [22]:
empiojar= False
empiojar_ruido= 0.25

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


## 1.3 Entrenamiento AutoGluon

La magia del Auto Machine Learning  aplicada a un dataset que posee MULTIPLES series de tiempo que se analizan en forma conjunta, suponiendo algun tipo de correlacion entre ellas.

AutoGluon TimeSeriesPredictor predicts future values of **multiple related** time series.

TimeSeriesPredictor provides probabilistic (quantile) multi-step-ahead orecasts for univariate time series. The forecast includes both the mean (i.e.,
 onditional expectation of future values given the past), as well as the quantiles of the forecast distribution, indicating the range of possible future outcomes.

TimeSeriesPredictor fits both “global” deep learning models that are **shared across all time series** (e.g., DeepAR, Transformer), as well as “local”
 statistical models that are fit to each individual time series (e.g., ARIMA, ETS).

TimeSeriesPredictor expects input data and makes predictions in the TimeSeriesDataFrame format.


https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [23]:
# NUEVO: columnas estáticas = todo lo de tb_productos menos product_id
cols_estaticas = [c for c in tb_ventas.columns if c not in ["product_id", "periodo", "timestamp", "tn"]]

static_df = tb_ventas.select(["product_id"] + cols_estaticas).unique("product_id").to_pandas()

ts_data = TimeSeriesDataFrame.from_data_frame(
  tb_ventas.select(["product_id", "timestamp", "tn"]).to_pandas(),
  timestamp_column='timestamp',
  id_column='product_id',
  static_features_df=static_df
)

Elegir cuidadosamente la metrica a utilizar
<br>Probar alternativas !
<br>Ese celda lleva 30 minutos en correr
<br>https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-metrics.html#forecasting-metrics
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.fit.html#autogluon.timeseries.TimeSeriesPredictor.fit
<br>https://auto.gluon.ai/stable/api/autogluon.timeseries.TimeSeriesPredictor.html

In [24]:
# Entrenamiento, esta es la parte pesada
global_eval_metric = 'RMSE' # alternativa RMSE

# defino
modelo = TimeSeriesPredictor(
  prediction_length= 2,  # horizonte de prediccion
  target= 'tn',
  freq= 'MS',  # Frecuencia mensual (Month Start)
  eval_metric= global_eval_metric
)

# entreno, le tira con muchisimos algorimtos, y prueba ensamblarlos
modelo.fit(ts_data,
  num_val_windows= 4,
  time_limit= 7200, # dos horas
  presets= "best_quality",  # Máxima calidad posible, obvio mayor tiempo de corrida
  random_seed= PARAM['semilla_primigenia']
)

Beginning AutoGluon training... Time limit = 7200s
AutoGluon will save models to '/content/.drive/My Drive/labo3/exp/AutoGluon-01/AutogluonModels/ag-20260605_173836'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
GPU Memory:         
Total GPU Memory:   Free: 0.00 GB, Allocated: 0.00 GB, Total: 0.00 GB
GPU Count:          0
Memory Avail:       10.97 GB / 12.67 GB (86.5%)
Disk Space Avail:   14.15 GB / 15.00 GB (94.3%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': RMSE,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 4,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/478M [00:00<?, ?B/s]

	-33.7091      = Validation score (-RMSE)
	79.25   s     = Training runtime
	16.28   s     = Validation (prediction) runtime
Training timeseries model Chronos2SmallFineTuned. Training for up to 1289.7s of the 7048.3s of remaining time.


config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/112M [00:00<?, ?B/s]

	-32.6654      = Validation score (-RMSE)
	1202.14 s     = Training runtime
	5.54    s     = Validation (prediction) runtime
Training timeseries model AutoETS. Training for up to 1310.1s of the 5840.5s of remaining time.
	-36.7095      = Validation score (-RMSE)
	64.22   s     = Training runtime
	25.43   s     = Validation (prediction) runtime
Training timeseries model ChronosWithRegressor[bolt_small]. Training for up to 1716.9s of the 5750.8s of remaining time.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

	-35.0720      = Validation score (-RMSE)
	25.29   s     = Training runtime
	4.68    s     = Validation (prediction) runtime
Training timeseries model TemporalFusionTransformer. Training for up to 2560.4s of the 5720.7s of remaining time.
	-31.4956      = Validation score (-RMSE)
	436.20  s     = Training runtime
	0.70    s     = Validation (prediction) runtime
Training timeseries model DeepAR. Training for up to 4683.7s of the 5283.7s of remaining time.
	-35.8219      = Validation score (-RMSE)
	299.60  s     = Training runtime
	2.71    s     = Validation (prediction) runtime
Fitting 1 ensemble(s), in 1 layers.
Training ensemble model WeightedEnsemble. Training for up to 4980.8s.
	Ensemble weights: {'AutoETS': 0.06, 'Chronos2SmallFineTuned': 0.17, 'DeepAR': 0.15, 'SeasonalNaive': 0.11, 'TemporalFusionTransformer': 0.51}
	-30.5637      = Validation score (-RMSE)
	3.50    s     = Training runtime
	35.65   s     = Validation (prediction) runtime
Training complete. Models trained: ['Seaso

## 1.4 Prediccion AutoGluon

In [25]:
# predict a partir los mismos datos

tb_forecast = modelo.predict(ts_data,
  random_seed= PARAM['semilla_primigenia']
)

display(tb_forecast)

data with frequency 'IRREG' has been resampled to frequency 'MS'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean          0.1          0.2          0.3  \
item_id timestamp                                                        
20001   2020-01-01  1309.813773   990.338575  1104.754140  1189.188232   
        2020-02-01  1360.659275  1016.694486  1139.377181  1231.453375   
20002   2020-01-01  1070.975876   762.301788   862.831576   954.466456   
        2020-02-01  1059.791324   732.938275   848.840364   938.413001   
20003   2020-01-01   735.226065   513.413773   591.347128   652.844501   
...                         ...          ...          ...          ...   
21266   2020-02-01     0.066527    -0.043678    -0.002833     0.022064   
21267   2020-01-01     0.040424    -0.032186    -0.004283     0.010795   
        2020-02-01     0.055350    -0.035186     0.000428     0.017611   
21276   2020-01-01     0.020451    -0.026612    -0.004995     0.005208   
        2020-02-01     0.032419    -0.021983     0.001290     0.011598   

                            0.4          0.5          0.6          0.7  \
item_id timestamp                                                        
20001   2020-01-01  1233.022707  1307.755992  1362.025278  1433.753292   
        2020-02-01  1286.904222  1361.762466  1422.706549  1494.300649   
20002   2020-01-01  1000.459218  1068.341676  1131.559922  1204.636979   
        2020-02-01   993.524160  1060.422339  1128.665479  1195.281259   
20003   2020-01-01   685.952521   736.534491   773.242800   817.473321   
...                         ...          ...          ...          ...   
21266   2020-02-01     0.043118     0.066811     0.088140     0.113594   
21267   2020-01-01     0.025489     0.040830     0.050726     0.068491   
        2020-02-01     0.036513     0.055479     0.070488     0.091312   
21276   2020-01-01     0.013907     0.023075     0.029643     0.039805   
        2020-02-01     0.021378     0.032897     0.040842     0.053565   

                            0.8          0.9  
item_id timestamp                             
20001   2020-01-01  1529.245973  1654.112984  
        2020-02-01  1599.970486  1740.532761  
20002   2020-01-01  1299.957357  1441.050801  
        2020-02-01  1305.612075  1458.132463  
20003   2020-01-01   877.841707   953.064921  
...                         ...          ...  
21266   2020-02-01     0.146024     0.200708  
21267   2020-01-01     0.086422     0.113616  
        2020-02-01     0.116895     0.153531  
21276   2020-01-01     0.054378     0.073768  
        2020-02-01     0.068103     0.089299  

[1560 rows x 10 columns]

In [26]:
# paso a formato Polars, teniendo en cuenta el indice
tb_forecast = pl.from_pandas(tb_forecast.reset_index())

In [27]:
# me quedo con ls predicciones de febrero-2020
# en 'mean' esta la prediccion, mas alla de los n-tiles
tb_final = tb_forecast.filter(pl.col("timestamp") ==  datetime(2020, 2, 1)).select(["item_id","mean"])

display(tb_final)

item_id,mean
i64,f64
20001,1360.659275
20002,1059.791324
20003,699.543154
20004,508.452855
20005,490.706898
…,…
21263,0.028101
21265,0.064021
21266,0.066527


## 1.5 Submit a Kaggle

In [28]:
# cambio nombre de campos a los que reconoce Kaggle
tb_final = tb_final.rename({
  "item_id": "product_id",
  "mean": "tn",
})

In [ ]:
# Submit a Kaggle
if not empiojar:
  archivo= "AutoGluon_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon " + global_eval_metric
else:
  archivo= "AutoGluon_empiojado_" + global_eval_metric + ".csv"
  mensaje= "AutoGluon logEMPIOJADO " + global_eval_metric + " al " + str(empiojar_ruido)

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )